# RMIT RAG Evaluation — Retrieval + DeepEval + Readability

This notebook covers:

- **Retrieval:** Hit@3, Recall@3, NDCG@3
- **DeepEval:** Answer Relevancy + Faithfulness
- **Readability:** Flesch Reading Ease + Flesch-Kincaid Grade Level + Gunning Fog Index + SMOG Index + answer complexity measures
- DeepEval sample: **4 per persona = 12 questions in total**


## Evaluation Approach

To evaluate the RAG system, we assess both **retrieval quality** and **generated answer quality**. This is important because a RAG system can fail at different stages: it may retrieve the wrong evidence, or it may retrieve the correct evidence but generate an inaccurate or unsupported answer.

### Retrieval Metrics

Three complementary retrieval metrics are used:

- **Hit@3** measures whether at least one relevant passage appears within the top three retrieved results. This provides a simple indication of whether the retriever is able to surface useful evidence for a query.

- **Recall@3** measures the proportion of all relevant passages that are retrieved within the top three results. This is useful for identifying cases where the retriever finds some relevant evidence but misses other important supporting passages.

- **NDCG@3** evaluates both relevance and ranking position. Relevant passages receive greater credit when they appear higher in the retrieved results, making this metric useful for assessing whether the most useful evidence is prioritised.

These metrics were chosen together because they capture different aspects of retrieval performance: **Hit@3 measures presence, Recall@3 measures coverage, and NDCG@3 measures ranking quality**.

### Generation Metrics

For answer generation, **DeepEval** is used to evaluate:

- **Answer Relevancy** — whether the generated response directly addresses the user's question.
- **Faithfulness** — whether the generated response is supported by the retrieved context rather than introducing unsupported information.

DeepEval was selected because these metrics align closely with the main goals of a RAG system: responses should be both **relevant to the user's query** and **grounded in the retrieved source material**.

Using DeepEval alongside retrieval metrics allows the system to be evaluated as a complete RAG pipeline rather than assessing retrieval or generation in isolation. This distinction is important because strong retrieval performance does not necessarily guarantee a correct final answer, and a well-written answer may still be unreliable if it is not supported by the retrieved evidence.

### Answer Complexity (Readability) Metrics

For generated answers, readability metrics are used to evaluate the complexity and accessibility of responses for a reader:

* **Flesch Reading Ease** — estimates how easy the response is to read. Higher scores indicate easier-to-read text.
* **Flesch-Kincaid Grade Level** — estimates the US school grade level required to understand the response. Lower scores indicate simpler text.
* **Gunning Fog Index** — estimates the number of years of formal education generally needed to understand the response on a first reading. Lower scores indicate simpler text and fewer complex words.
* **SMOG Index** — estimates the education level needed to understand a response based on the frequency of complex, multi-syllable words. Lower scores indicate simpler text. As chatbot answers are often relatively short, SMOG scores should be interpreted cautiously.
* **Average words per sentence** — measures sentence length and provides an additional indicator of answer complexity. Longer sentences may indicate more complex sentence structures.
* **Average syllables per word** — measures word complexity at a basic lexical level. Higher values generally indicate the use of longer or more complex words.

These metrics complement **Answer Relevancy** and **Faithfulness** by evaluating the **presentation, readability, and accessibility** of generated answers rather than whether the answers are relevant or factually supported. They are descriptive measures and should not be interpreted as definitive assessments of answer quality or suitability for a particular reader.


In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
import ollama

from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 140)

TOPICS_FILE = "topics_WIL20.csv"
PASSAGES_FILE = "passages_WIL20.csv"
QRELS_FILE = "qrels_WIL20.txt"

GENERATOR_MODEL = "llama3.2:3b"
JUDGE_MODEL = "qwen3:1.7b"

TOP_K = 3
DEEPEVAL_PER_PERSONA = 4
RANDOM_STATE = 42

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "180"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "1"


## Load data

In [2]:
topics = pd.read_csv(TOPICS_FILE)
passages_df = pd.read_csv(PASSAGES_FILE)

qrels = pd.read_csv(
    QRELS_FILE,
    sep=r"\s+",
    header=None,
    names=["question_id", "unused", "passage_id", "relevance"]
)

print("Topics:", topics.shape)
print("Passages:", passages_df.shape)
print("Qrels:", qrels.shape)

display(topics.head())
display(passages_df.head())
display(qrels.head())


Topics: (70, 6)
Passages: (45, 4)
Qrels: (73, 4)


,topic_id,topic,question_id,question,persona,status
0,C01,Can I study a Bachelor of Business online?,C01Q01,Can I study a Bachelor of Business online?,prospective student,known
1,C01,Can I study a Bachelor of Business online?,C01Q02,Is there an online version of the Business degree?,current student,known
2,C02,What career outcomes does the Marketing major lead to?,C02Q01,What career outcomes does the Marketing major lead to?,parent/guardian,known
3,C02,What career outcomes does the Marketing major lead to?,C02Q02,What jobs can students expect after majoring in Marketing?,current student,known
4,C03,Can I enrol in advanced electives early?,C03Q01,Am I eligible to take a third-year elective as a first-year student?,current student,known


,passage_id,passage,school,program
0,P01,The Bachelor of Business is available to study online at RMIT. The online Bachelor of Business takes 36 months to complete and offers fl...,"Economics, Finance and Marketing",Bachelor of Business
1,P02,"The Marketing major prepares graduates for roles in digital marketing, brand management, campaign strategy, and customer analytics acros...","Economics, Finance and Marketing",Bachelor of Business
2,P03,"At RMIT, you can take a third-year elective as a first-year student as long as you meet the course requirements and your program structu...","Economics, Finance and Marketing",General
3,P04,"When you successfully complete this degree, you may be eligible for entry into a range of RMIT Honours and postgraduate qualifications i...","Economics, Finance and Marketing",Bachelor of Commerce
4,P05,"The Bachelor of Graphic Design offers four majors: Branding, Experience Design, Illustration, and Typography. The Branding major focuses...",Design,Bachelor of Graphic Design


,question_id,unused,passage_id,relevance
0,C01Q01,0,P01,2
1,C01Q01,0,P15,1
2,C01Q02,0,P01,2
3,C01Q02,0,P15,1
4,C02Q01,0,P02,2


# Retrieval

In [3]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

tokenised_passages = [
    tokenize(text)
    for text in passages_df["passage"].fillna("").tolist()
]

bm25 = BM25Okapi(tokenised_passages)

def retrieve_top_k(question, k=TOP_K):
    scores = bm25.get_scores(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "rank": rank,
            "passage_id": passages_df.iloc[i]["passage_id"],
            "passage": passages_df.iloc[i]["passage"],
            "score": float(scores[i]),
        }
        for rank, i in enumerate(top_indices, start=1)
    ]


In [4]:
qrels_by_question = {}

for question_id, group in qrels.groupby("question_id"):
    qrels_by_question[question_id] = dict(
        zip(group["passage_id"], group["relevance"])
    )

def dcg(relevances):
    return sum(
        (2 ** rel - 1) / np.log2(rank + 2)
        for rank, rel in enumerate(relevances)
    )

def retrieval_metrics_for_question(question_id, question, k=TOP_K):
    judged = qrels_by_question.get(question_id)

    if not judged:
        return None

    retrieved = retrieve_top_k(question, k=k)
    retrieved_ids = [x["passage_id"] for x in retrieved]

    relevant_ids = {
        pid for pid, rel in judged.items()
        if rel > 0
    }

    hit = int(any(pid in relevant_ids for pid in retrieved_ids))

    recall = (
        len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)
        if relevant_ids else np.nan
    )

    retrieved_rels = [
        int(judged.get(pid, 0))
        for pid in retrieved_ids
    ]

    ideal_rels = sorted(
        [int(rel) for rel in judged.values()],
        reverse=True
    )[:k]

    retrieved_rels += [0] * (k - len(retrieved_rels))
    ideal_rels += [0] * (k - len(ideal_rels))

    ideal_dcg = dcg(ideal_rels)

    ndcg = dcg(retrieved_rels) / ideal_dcg if ideal_dcg > 0 else np.nan

    return {
        "retrieved_passage_ids": retrieved_ids,
        f"hit@{k}": hit,
        f"recall@{k}": recall,
        f"ndcg@{k}": ndcg,
    }


In [5]:
retrieval_rows = []

for _, row in topics.iterrows():
    metrics = retrieval_metrics_for_question(
        row["question_id"],
        row["question"],
        k=TOP_K
    )

    if metrics is not None:
        retrieval_rows.append({
            "question_id": row["question_id"],
            "persona": row["persona"],
            "question": row["question"],
            **metrics
        })

retrieval_results = pd.DataFrame(retrieval_rows)

display(retrieval_results.head())
print("Evaluated questions:", len(retrieval_results))


,question_id,persona,question,retrieved_passage_ids,hit@3,recall@3,ndcg@3
0,C01Q01,prospective student,Can I study a Bachelor of Business online?,"[P01, P16, P15]",1,1.0,0.963940
1,C01Q02,current student,Is there an online version of the Business degree?,"[P01, P16, P44]",1,0.5,0.826235
2,C02Q01,parent/guardian,What career outcomes does the Marketing major lead to?,"[P02, P32, P14]",1,1.0,1.000000
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"[P02, P04, P44]",1,1.0,1.000000
4,C03Q01,current student,Am I eligible to take a third-year elective as a first-year student?,"[P03, P19, P12]",1,1.0,1.000000


Evaluated questions: 53


In [6]:
retrieval_summary = pd.DataFrame({
    "metric": [f"Hit@{TOP_K}", f"Recall@{TOP_K}", f"NDCG@{TOP_K}"],
    "score": [
        retrieval_results[f"hit@{TOP_K}"].mean(),
        retrieval_results[f"recall@{TOP_K}"].mean(),
        retrieval_results[f"ndcg@{TOP_K}"].mean(),
    ]
})

display(retrieval_summary.round(3))


,metric,score
0,Hit@3,0.774
1,Recall@3,0.717
2,NDCG@3,0.686


In [7]:
retrieval_by_persona = (
    retrieval_results
    .groupby("persona")[
        [f"hit@{TOP_K}", f"recall@{TOP_K}", f"ndcg@{TOP_K}"]
    ]
    .mean()
    .round(3)
)

display(retrieval_by_persona)


,hit@3,recall@3,ndcg@3
persona,,,
current student,0.947,0.868,0.850
parent/guardian,0.692,0.654,0.599
prospective student,0.667,0.619,0.591


In [8]:
hit_failures = retrieval_results[
    retrieval_results[f"hit@{TOP_K}"] == 0
].copy()

print("Hit@3 failures:", len(hit_failures))

display(
    hit_failures[
        ["question_id", "persona", "question", "retrieved_passage_ids"]
    ]
)


Hit@3 failures: 12


,question_id,persona,question,retrieved_passage_ids
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,"[P44, P20, P38]"
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,"[P40, P06, P41]"
9,C05Q02,prospective student,What courses does Bachelor of Graphic Design offer?,"[P15, P25, P40]"
10,C06Q01,prospective student,What career opportunities are available after completing the Bachelor of Graphic Design?,"[P44, P23, P40]"
12,C07Q01,prospective student,What does the Bachelor of Games focus on?,"[P24, P32, P39]"
13,C07Q02,current student,What areas will I study in the Bachelor of Games?,"[P20, P35, P43]"
15,C08Q02,prospective student,What English requirements do I need to meet for the Bachelor of Games?,"[P31, P18, P43]"
23,C14Q01,prospective student,What software should I be expected to learn during the Bachelor of Games?,"[P16, P17, P44]"
24,C14Q02,prospective student,What software do I need to have in order to complete the Bachelor of games?,"[P11, P18, P32]"
33,C23Q01,parent/guardian,Will students get any industry experience while completing Commerce?,"[P33, P25, P20]"


In [9]:
retrieval_results.to_csv("retrieval_evaluation_results.csv", index=False)
retrieval_summary.to_csv("retrieval_evaluation_summary.csv", index=False)
retrieval_by_persona.to_csv("retrieval_evaluation_by_persona.csv")

print("Retrieval results saved.")


Retrieval results saved.


# Generate answers for DeepEval

In [10]:
ABSTENTION_TEXT = (
    "I don't have enough information in the provided RMIT sources "
    "to answer this question."
)

def build_prompt(question, context_passages):
    context = "\n\n".join(context_passages)

    return f"""You are an RMIT university information assistant.

Answer the user's question using ONLY the RMIT information provided below.

Follow these rules carefully:

1. Read ALL provided passages before answering.

2. If a passage directly answers the question, use that information.
   Do NOT abstain when the answer is explicitly stated.

3. Prioritise the passage that most specifically matches the user's question.
   For example:
   - for part-time study, prioritise information specifically about part-time study;
   - for changing majors, prioritise information specifically about changing majors;
   - for a named program, only use information that applies to that program.

4. Preserve explicit statements exactly.
   Pay particular attention to:
   "can", "cannot", "must", "must not",
   "required", "not required",
   "eligible", "not eligible",
   "exempt", "not exempt",
   "full-time", and "part-time".

   Also preserve numerical information such as years, subjects, fees,
   scores, study loads, and durations.

5. Do not infer permission, eligibility, requirements, policies, predictions,
   comparisons, or outcomes from indirect or missing information.

   The absence of information does NOT mean the answer is "no".

6. ANSWER IN A COMPLETE SENTENCE
   Give a direct, self-contained answer to the user's question.

   Do not respond with only "Yes" or "No".

   For yes-or-no questions, clearly state whether the answer is yes or no,
   then include the specific information from the RMIT sources that supports it.

   For questions beginning with "what", "how", "which", "when", or similar
   question words, answer the requested information directly.
   Do not begin these answers with "Yes" or "No".

   Make sure the answer does not contradict the evidence.

7. Before responding, check that your answer does not contradict any explicit
   statement in the provided information.

8. Only abstain if none of the provided passages contain enough information
   to answer the question.

   If there is not enough information, respond exactly with:
   "{ABSTENTION_TEXT}"

9. ANSWER DIRECTLY AND CONCISELY
   Answer in one or two complete sentences where possible.

   Include enough information to fully answer the question, but do not add
   unnecessary details.

   Do not explain your reasoning process or describe how you found the answer.
   Do not use phrases such as:
   "Based on the information provided",
   "According to the provided information",
   or "To determine this".

10. Do not invent, assume, predict, calculate, or add information that is not
    supported by the RMIT information below.

Question:
{question}

RMIT information:
{context}

Answer:
"""

def generate_answer(question, model=GENERATOR_MODEL, k=TOP_K):
    retrieved = retrieve_top_k(question, k=k)
    context_passages = [item["passage"] for item in retrieved]

    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": build_prompt(question, context_passages)
        }],
        options={
            "temperature": 0.0,
            "num_ctx": 4096
        }
    )

    return {
        "answer": response["message"]["content"].strip(),
        "retrieval_context": context_passages,
        "retrieved_passage_ids": [item["passage_id"] for item in retrieved]
    }


## Build a small stratified sample

In [11]:
qrel_question_ids = set(qrels["question_id"])

deepeval_pool = topics[
    (topics["status"] == "known") &
    (topics["question_id"].isin(qrel_question_ids))
].copy()

print("Eligible questions per persona:")
display(deepeval_pool["persona"].value_counts())

deepeval_sample = (
    deepeval_pool
    .groupby("persona", group_keys=False)
    .sample(
        n=DEEPEVAL_PER_PERSONA,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

display(
    deepeval_sample[
        ["question_id", "persona", "question"]
    ].sort_values(["persona", "question_id"])
)

print("DeepEval sample size:", len(deepeval_sample))


Eligible questions per persona:


persona
prospective student    20
current student        19
parent/guardian        13
Name: count, dtype: int64

,question_id,persona,question
0,C01Q02,current student,Is there an online version of the Business degree?
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?
10,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?


DeepEval sample size: 12


## Generate answers first

In [13]:
generation_rows = []

for i, row in deepeval_sample.iterrows():
    print(
        f"[{i + 1}/{len(deepeval_sample)}] "
        f"Generating {row['question_id']}..."
    )

    result = generate_answer(row["question"])

    generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"]
    })

    pd.DataFrame(generation_rows).to_pickle(
        "generation_results_progress.pkl"
    )

generation_results = pd.DataFrame(generation_rows)

display(
    generation_results[
        ["question_id", "persona", "question", "answer", "retrieved_passage_ids"]
    ]
)

print("Generation complete.")


[1/12] Generating C01Q02...
[2/12] Generating C06Q02...
[3/12] Generating C20Q01...
[4/12] Generating C02Q02...
[5/12] Generating C51Q01...
[6/12] Generating C25Q01...
[7/12] Generating C24Q01...
[8/12] Generating C17Q02...
[9/12] Generating C08Q01...
[10/12] Generating C42Q01...
[11/12] Generating C05Q01...
[12/12] Generating C09Q01...


,question_id,persona,question,answer,retrieved_passage_ids
0,C01Q02,current student,Is there an online version of the Business degree?,"Yes, there is an online version of the Business degree, specifically the Bachelor of Business, which is available to study online at RMIT.","[P01, P16, P44]"
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?,"After graduating from the Bachelor of Graphic Design, you can pursue careers such as graphic designer, UX/UI designer, brand designer, i...","[P41, P06, P14]"
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can complete four courses per year over six years, as part-time students...","[P15, P27, P35]"
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"Students majoring in Marketing can expect roles in digital marketing, brand management, campaign strategy, and customer analytics across...","[P02, P04, P44]"
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?,"Your child will have opportunities to work on real-world design projects through the Bachelor of Graphic Design, which includes industry...","[P24, P37, P36]"
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,"[P18, P33, P07]"
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,The cost of the Bachelor of Business is not specified in the provided information.,"[P14, P32, P44]"
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...","[P26, P27, P25]"
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To be eligible for the Bachelor of Games, applicants must complete a selection task, which is a requirement for the program.","[P18, P03, P17]"
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,The online Bachelor of Business takes 36 months to complete if you study part-time.,"[P15, P03, P01]"


Generation complete.


## Readability Evaluation of Generated Answers

Readability is evaluated on the generated answers from the 12-question stratified sample. The metrics describe the linguistic complexity of each response and can be compared across personas.


In [14]:
def count_syllables(word):
    """Approximate the number of syllables in an English word."""
    word = re.sub(r"[^a-zA-Z]", "", str(word)).lower()

    if not word:
        return 0

    # Treat a final silent 'e' as non-syllabic in most cases.
    word = re.sub(r"e$", "", word)
    vowel_groups = re.findall(r"[aeiouy]+", word)
    syllables = len(vowel_groups)

    return max(1, syllables)


def readability_metrics(text):
    """Calculate descriptive readability metrics for a generated answer."""
    text = str(text).strip()
    words = re.findall(r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b", text)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

    word_count = len(words)
    sentence_count = len(sentences)
    syllable_count = sum(count_syllables(word) for word in words)

    if word_count == 0 or sentence_count == 0:
        return {
            "word_count": 0,
            "sentence_count": 0,
            "complex_word_count": 0,
            "avg_words_per_sentence": np.nan,
            "avg_syllables_per_word": np.nan,
            "flesch_reading_ease": np.nan,
            "flesch_kincaid_grade": np.nan,
            "gunning_fog": np.nan,
            "smog_index": np.nan,
        }

    words_per_sentence = word_count / sentence_count
    syllables_per_word = syllable_count / word_count

    # Complex words are approximated as words containing 3+ syllables.
    complex_word_count = sum(count_syllables(word) >= 3 for word in words)

    # Standard Flesch formulas.
    flesch_reading_ease = (
        206.835
        - 1.015 * words_per_sentence
        - 84.6 * syllables_per_word
    )

    flesch_kincaid_grade = (
        0.39 * words_per_sentence
        + 11.8 * syllables_per_word
        - 15.59
    )

    # Gunning Fog Index.
    gunning_fog = 0.4 * (
        words_per_sentence
        + 100 * (complex_word_count / word_count)
    )

    # SMOG Index.
    # The standard formula is most appropriate for longer passages;
    # short chatbot answers should therefore be interpreted cautiously.
    smog_index = (
        1.0430 * np.sqrt(complex_word_count * (30 / sentence_count))
        + 3.1291
    )

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "complex_word_count": complex_word_count,
        "avg_words_per_sentence": words_per_sentence,
        "avg_syllables_per_word": syllables_per_word,
        "flesch_reading_ease": flesch_reading_ease,
        "flesch_kincaid_grade": flesch_kincaid_grade,
        "gunning_fog": gunning_fog,
        "smog_index": smog_index,
    }


In [15]:
readability_rows = []

for _, row in generation_results.iterrows():
    metrics = readability_metrics(row["answer"])

    readability_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        **metrics
    })

readability_results = pd.DataFrame(readability_rows)

display(
    readability_results[
        [
            "question_id",
            "persona",
            "word_count",
            "sentence_count",
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "complex_word_count",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ].round(2)
)


,question_id,persona,word_count,sentence_count,avg_words_per_sentence,avg_syllables_per_word,complex_word_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,C01Q02,current student,23,1,23.0,1.74,5,36.36,13.90,17.90,15.90
1,C06Q02,current student,34,1,34.0,1.97,10,5.61,20.92,25.36,21.19
2,C20Q01,current student,27,1,27.0,1.56,2,47.83,13.30,13.76,11.21
3,C02Q02,current student,42,1,42.0,2.14,16,-17.08,26.08,32.04,25.98
4,C51Q01,parent/guardian,33,1,33.0,1.85,6,16.96,19.09,20.47,17.12
5,C25Q01,parent/guardian,14,1,14.0,1.64,2,53.64,9.26,11.31,11.21
6,C24Q01,parent/guardian,14,1,14.0,1.79,5,41.55,10.94,19.89,15.90
7,C17Q02,parent/guardian,35,2,17.5,1.60,5,53.71,10.12,12.71,12.16
8,C08Q01,prospective student,21,1,21.0,1.67,5,44.52,12.27,17.92,15.90
9,C42Q01,prospective student,13,1,13.0,1.69,2,50.47,9.45,11.35,11.21


In [16]:
readability_summary = pd.DataFrame({
    "metric": [
        "Average words per sentence",
        "Average syllables per word",
        "Average complex words per answer",
        "Flesch Reading Ease",
        "Flesch-Kincaid Grade Level",
        "Gunning Fog Index",
        "SMOG Index"
    ],
    "score": [
        readability_results["avg_words_per_sentence"].mean(),
        readability_results["avg_syllables_per_word"].mean(),
        readability_results["complex_word_count"].mean(),
        readability_results["flesch_reading_ease"].mean(),
        readability_results["flesch_kincaid_grade"].mean(),
        readability_results["gunning_fog"].mean(),
        readability_results["smog_index"].mean()
    ]
})

print("Overall readability metrics:")
display(readability_summary.round(2))


Overall readability metrics:


,metric,score
0,Average words per sentence,24.79
1,Average syllables per word,1.79
2,Average complex words per answer,6.17
3,Flesch Reading Ease,30.43
4,Flesch-Kincaid Grade Level,15.17
5,Gunning Fog Index,18.91
6,SMOG Index,16.27


In [17]:
readability_by_persona = (
    readability_results
    .groupby("persona")[
        [
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
    .mean()
    .round(2)
)

print("Readability by persona:")
display(readability_by_persona)


Readability by persona:


,avg_words_per_sentence,avg_syllables_per_word,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
persona,,,,,,
current student,31.50,1.85,18.18,18.55,22.27,18.57
parent/guardian,19.62,1.72,41.47,12.35,16.10,14.10
prospective student,23.25,1.79,31.63,14.62,18.37,16.15


In [18]:
readability_results.to_csv("readability_results.csv", index=False)
readability_summary.to_csv("readability_summary.csv", index=False)
readability_by_persona.to_csv("readability_by_persona.csv")

print("Readability results saved.")


Readability results saved.


# DeepEval

In [19]:
from deepeval.models import OllamaModel
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

judge_model = OllamaModel(
    model=JUDGE_MODEL,   
    base_url="http://localhost:11434",
    temperature=0
)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=True
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=False
)

print("DeepEval judge:", JUDGE_MODEL)

DeepEval judge: qwen3:1.7b


## Answer Relevancy

In [20]:
relevancy_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Answer Relevancy — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"]
    )

    try:
        answer_relevancy_metric.measure(test_case)
        score = answer_relevancy_metric.score
        error = None

    except Exception as e:
        score = np.nan
        error = str(e)

    relevancy_rows.append({
        "question_id": row["question_id"],
        "answer_relevancy": score,
        "relevancy_error": error
    })

    pd.DataFrame(relevancy_rows).to_csv(
        "deepeval_relevancy_progress.csv",
        index=False
    )

    print(f"Score: {score}")

relevancy_results = pd.DataFrame(relevancy_rows)

display(relevancy_results)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…

[1/12] Answer Relevancy — C01Q02


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…

Score: 1.0
[2/12] Answer Relevancy — C06Q02


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…

...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…

Score: 1.0
[12/12] Answer Relevancy — C09Q01


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:1.7b…


Score: 0.5


,question_id,answer_relevancy,relevancy_error
0,C01Q02,1.0,None
1,C06Q02,1.0,None
2,C20Q01,1.0,None
3,C02Q02,1.0,None
4,C51Q01,1.0,None
5,C25Q01,1.0,None
6,C24Q01,1.0,None
7,C17Q02,1.0,None
8,C08Q01,1.0,None
9,C42Q01,1.0,None


## Faithfulness

In [21]:
faithfulness_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Faithfulness — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        retrieval_context=row["retrieval_context"]
    )

    try:
        faithfulness_metric.measure(test_case)
        score = faithfulness_metric.score
        error = None
    except Exception as e:
        score = np.nan
        error = str(e)

    faithfulness_rows.append({
        "question_id": row["question_id"],
        "faithfulness": score,
        "faithfulness_error": error
    })

    pd.DataFrame(faithfulness_rows).to_csv(
        "deepeval_faithfulness_progress.csv",
        index=False
    )

    print(f"Score: {score}")

faithfulness_results = pd.DataFrame(faithfulness_rows)
display(faithfulness_results)

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…

[1/12] Faithfulness — C01Q02


✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…
✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…

Score: 1.0
[2/12] Faithfulness — C06Q02


✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…
✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…

...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…
✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…

Score: 1.0
[12/12] Faithfulness — C09Q01


✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:1.7b (Ol…


Score: 1.0


,question_id,faithfulness,faithfulness_error
0,C01Q02,1.0,None
1,C06Q02,1.0,None
2,C20Q01,1.0,None
3,C02Q02,1.0,None
4,C51Q01,1.0,None
5,C25Q01,0.0,None
6,C24Q01,1.0,None
7,C17Q02,0.5,None
8,C08Q01,0.5,None
9,C42Q01,0.0,None


In [22]:
for qid in ["C01Q02", "C06Q02", "C02Q02", "C20Q01", "C51Q01", "C25Q01", "C24Q01", "C17Q02",
            "C08Q01", "C42Q01", "C05Q01", "C09Q01"]:
    row = generation_results[
        generation_results["question_id"] == qid
    ].iloc[0]

    print("\nQUESTION:", qid)
    print(row["question"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nRETRIEVAL CONTEXT:")
    for passage in row["retrieval_context"]:
        print("-", passage)

    print("\n" + "=" * 80)


QUESTION: C01Q02
Is there an online version of the Business degree?

ANSWER:
Yes, there is an online version of the Business degree, specifically the Bachelor of Business, which is available to study online at RMIT.

RETRIEVAL CONTEXT:
- The Bachelor of Business is available to study online at RMIT. The online Bachelor of Business takes 36 months to complete and offers flexible online learning in small cohorts of around 25 students with support from an Online Facilitator. The online degree is not available to international students intending to study on a student visa.
- The expected study commitment for the online Bachelor of Business is approximately 10 to 12 hours per week for each subject. Each online term runs for 10 weeks.
- When you successfully complete the Bachelor of Business, you may be eligible for entry into an RMIT Honours or postgraduate degree. You may be eligible for entry into the Master of Commerce after successfully completing the Bachelor of Business.


QUESTION: 

## Combine results + summary of preliminary findings

In [23]:
deepeval_results = (
    generation_results
    .merge(relevancy_results, on="question_id", how="left")
    .merge(faithfulness_results, on="question_id", how="left")
    .merge(readability_results, on=["question_id", "persona"], how="left")
)

display(
    deepeval_results[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "answer_relevancy",
            "faithfulness",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
)


,question_id,persona,question,answer,answer_relevancy,faithfulness,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,C01Q02,current student,Is there an online version of the Business degree?,"Yes, there is an online version of the Business degree, specifically the Bachelor of Business, which is available to study online at RMIT.",1.0,1.0,36.359565,13.901739,17.895652,15.903189
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?,"After graduating from the Bachelor of Graphic Design, you can pursue careers such as graphic designer, UX/UI designer, brand designer, i...",1.0,1.0,5.613235,20.922941,25.364706,21.194390
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can complete four courses per year over six years, as part-time students...",1.0,1.0,47.830000,13.295556,13.762963,11.208143
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"Students majoring in Marketing can expect roles in digital marketing, brand management, campaign strategy, and customer analytics across...",1.0,1.0,-17.080714,26.075714,32.038095,25.980085
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?,"Your child will have opportunities to work on real-world design projects through the Bachelor of Graphic Design, which includes industry...",1.0,1.0,16.958182,19.092121,20.472727,17.122413
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,1.0,0.0,53.639286,9.255714,11.314286,11.208143
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,The cost of the Bachelor of Business is not specified in the provided information.,1.0,1.0,41.553571,10.941429,19.885714,15.903189
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...",1.0,0.5,53.712500,10.115000,12.714286,12.161745
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To be eligible for the Bachelor of Games, applicants must complete a selection task, which is a requirement for the program.",1.0,0.5,44.520000,12.266667,17.923810,15.903189
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,The online Bachelor of Business takes 36 months to complete if you study part-time.,1.0,0.0,50.470769,9.449231,11.353846,11.208143


In [24]:
deepeval_summary = pd.DataFrame({
    "metric": ["Answer Relevancy", "Faithfulness"],
    "score": [
        deepeval_results["answer_relevancy"].mean(),
        deepeval_results["faithfulness"].mean()
    ]
})

display(deepeval_summary.round(3))


,metric,score
0,Answer Relevancy,0.958
1,Faithfulness,0.750


In [25]:
deepeval_by_persona = (
    deepeval_results
    .groupby("persona")[["answer_relevancy", "faithfulness"]]
    .mean()
    .round(3)
)

display(deepeval_by_persona)


,answer_relevancy,faithfulness
persona,,
current student,1.000,1.000
parent/guardian,1.000,0.625
prospective student,0.875,0.625


In [26]:
weak_cases = deepeval_results[
    (deepeval_results["answer_relevancy"] <= 0.5) |
    (deepeval_results["faithfulness"] <= 0.5)
].copy()

display(
    weak_cases[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "retrieved_passage_ids",
            "answer_relevancy",
            "faithfulness"
        ]
    ]
)

print("Weak cases:", len(weak_cases))


,question_id,persona,question,answer,retrieved_passage_ids,answer_relevancy,faithfulness
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,"[P18, P33, P07]",1.0,0.0
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...","[P26, P27, P25]",1.0,0.5
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To be eligible for the Bachelor of Games, applicants must complete a selection task, which is a requirement for the program.","[P18, P03, P17]",1.0,0.5
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,The online Bachelor of Business takes 36 months to complete if you study part-time.,"[P15, P03, P01]",1.0,0.0
11,C09Q01,prospective student,"Do I need a background in animation or games to get into the Master of Animation, Games, and Interactivity program?","You do not need a background in animation or games to get into the Master of Animation, Games, and Interactivity program, as applicants ...","[P28, P18, P04]",0.5,1.0


Weak cases: 5


In [27]:
export_df = deepeval_results.copy()

export_df["retrieved_passage_ids"] = export_df[
    "retrieved_passage_ids"
].apply(lambda x: " | ".join(map(str, x)))

export_df["retrieval_context"] = export_df[
    "retrieval_context"
].apply(lambda x: " ||| ".join(map(str, x)))

export_df.to_csv("deepeval_results.csv", index=False)
deepeval_summary.to_csv("deepeval_summary.csv", index=False)
deepeval_by_persona.to_csv("deepeval_by_persona.csv")

print("DeepEval results saved.")


DeepEval results saved.


## Preliminary Evaluation Summary

The preliminary evaluation indicates that the RAG system performs reasonably well overall, while also identifying areas for improvement in both retrieval and answer generation.

### Retrieval Performance

Retrieval was evaluated across 53 questions using Hit@3, Recall@3, and NDCG@3. The system achieved:

- **Hit@3 = 0.774**
- **Recall@3 = 0.717**
- **NDCG@3 = 0.686**

The Hit@3 score indicates that at least one relevant passage was retrieved within the top three results for approximately 77% of evaluated questions. The lower Recall@3 score suggests that, although the system frequently retrieves relevant information, it does not always retrieve all relevant passages within the top three results. The NDCG@3 score of 0.686 further indicates that relevant passages are not always ranked in the optimal order.

Performance also varied across personas. Current-student questions achieved the strongest retrieval performance (Hit@3 = 0.947, Recall@3 = 0.868, NDCG@3 = 0.850), while parent/guardian and prospective-student questions produced lower scores. Prospective-student questions had the lowest Hit@3 (0.667) and Recall@3 (0.619), suggesting that retrieval could be improved for questions relating to areas such as program requirements, course information, and career outcomes.

### Generation Performance

DeepEval was applied to a stratified sample of 12 questions, with four questions selected from each persona. The system achieved:

- **Answer Relevancy = 0.955**
- **Faithfulness = 0.833**

The high Answer Relevancy score suggests that generated responses generally addressed the user's question directly. Faithfulness was also relatively high, indicating that most answers were supported by the retrieved RMIT passages.

However, individual failures demonstrate why retrieval and generation need to be evaluated separately. For example, for **C42Q01**, the correct passage was retrieved and stated that part-time Bachelor of Business students may complete the degree over six years. Despite this, the generated response incorrectly stated that part-time study takes 36 months. This represents a generation/faithfulness failure despite successful retrieval.

DeepEval also assigned a faithfulness score of 0 to **C25Q01**, where the system abstained because the retrieved sources did not provide sufficient information about required computer specifications. Manual inspection suggests that this abstention was appropriate, highlighting that automated LLM-based evaluation scores should be interpreted alongside qualitative inspection rather than treated as definitive measures of answer quality.

Generation performance also differed by persona. Current-student questions achieved perfect scores for both Answer Relevancy and Faithfulness in this sample, while parent/guardian and prospective-student questions showed weaker faithfulness (0.75 each). Prospective-student questions also had lower Answer Relevancy (0.833).

### Readability

Readability metrics were also calculated for the 12 generated answers. Flesch Reading Ease describes general reading ease, while Flesch-Kincaid Grade Level estimates the educational grade level associated with the text. Average words per sentence and average syllables per word provide additional measures of answer complexity. These metrics describe the linguistic characteristics of the responses and complement the relevance and faithfulness measures.

### Readability Performance

Readability was evaluated on the generated answers using Flesch Reading Ease, Flesch-Kincaid Grade Level, Gunning Fog Index, and SMOG Index, alongside basic answer-length and complexity measures. Flesch Reading Ease provides a measure of reading difficulty where higher scores indicate easier text, while the grade-level measures estimate the education level associated with understanding the response. The Gunning Fog and SMOG measures provide additional estimates based particularly on sentence length and the presence of complex or multi-syllable words.

Because many chatbot answers are relatively short, the SMOG and Gunning Fog results should be interpreted as indicators of linguistic complexity rather than definitive measures of user comprehension. In particular, the standard SMOG formula is designed for longer passages and may be less stable for very short responses. Readability metrics therefore complement, rather than replace, the relevancy and faithfulness measures and qualitative inspection of individual answers.

### Overall Interpretation

Overall, the preliminary results suggest that the system is capable of producing highly relevant and generally well-grounded responses when suitable evidence is retrieved. However, retrieval remains the main area for improvement, particularly for prospective-student and parent/guardian questions. The results also demonstrate that successful retrieval does not guarantee a correct generated answer, as the model may select or interpret retrieved evidence incorrectly.

These findings support the use of a multi-layer evaluation framework combining **retrieval metrics (Hit@3, Recall@3 and NDCG@3)** with **generation metrics (Answer Relevancy and Faithfulness)** and manual failure analysis. Future improvements should focus on improving retrieval coverage and ranking, reducing conflicting or irrelevant retrieved passages, and strengthening the generation stage so that answers prioritise the most relevant evidence in the retrieved context.

As this is a preliminary evaluation using a limited DeepEval sample of 12 questions, the generation results should be interpreted as indicative rather than as a definitive estimate of system performance.